# Criando Sessão e configs

In [1]:
import os

java_home = "/home/bruno/.jdk/jdk-17.0.19+10"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local")
    .appName("PySpark_01")
    .getOrCreate()
)
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/03 17:44:57 WARN Utils: Your hostname, bruno-B550M-AORUS-ELITE, resolves to a loopback address: 127.0.1.1; using 192.168.4.2 instead (on interface enp4s0)
26/07/03 17:44:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 17:44:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.1.2


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import re

In [3]:
spark = (
    SparkSession.builder
    .master("local")
    .appName("PySpark_01")
    .getOrCreate()
)

print(spark.version)



4.1.2


In [4]:
df = spark.read.parquet("/home/bruno/projeto_clima-df/data/raw_df/INMET_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025.parquet", header=True, inferSchema=True)
df.show(3)

26/07/03 17:45:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+---+----------+------------+------------+-----------+--------+----------------+----------+--------+--------------------------------+-----------------------------------------------------+-----------------------------------------------+------------------------------------------------+-----------------------+--------------------------------------------+------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------------+------------------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------+------------------------------------+--------------------------+-------------------------------+
|REGIAO| UF|   ESTACAO|CODIGO (WMO)|    LATITUDE|  LONGITUDE|ALTITUDE|DATA DE FUNDACAO|      Data|Hora UTC|PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|PRESSÃO AT

# Comandos para  ajuda (Lembrar de remover)

In [5]:
"""df.createOrReplaceTempView("frame")
teste = spark.sql("select distinct(pressao_max_mb) from frame").show() """

'df.createOrReplaceTempView("frame")\nteste = spark.sql("select distinct(pressao_max_mb) from frame").show() '

# Removendo o "." do nome das colunas

In [6]:
df = df.toDF(*[col.replace(".", "") for col in df.columns])

# Renomeando Colunas

In [8]:
padronizar_colunas = [
    col.lower().replace(",","_").replace(" ", "_")
    for col in df.columns
]

renames = {
    "precipitação_total__horário_(mm)": "precipitacao_total_mm",
    "pressao_atmosferica_ao_nivel_da_estacao__horaria_(mb)": "pressao_estacao_mb",
    "pressão_atmosferica_maxna_hora_ant_(aut)_(mb)": "pressao_max_mb",
    "pressão_atmosferica_min_na_hora_ant_(aut)_(mb)": "pressao_min_mb",
    "radiacao_global_(kj/m²)": "radiacao_global_kj_m2",
    "temperatura_do_ar_-_bulbo_seco__horaria_(°c)": "temperatura_seco_c",
    "temperatura_do_ponto_de_orvalho_(°c)": "temperatura_orvalho_c",
    "temperatura_máxima_na_hora_ant_(aut)_(°c)": "temperatura_max_c",
    "temperatura_mínima_na_hora_ant_(aut)_(°c)": "temperatura_min_c",
    "temperatura_orvalho_max_na_hora_ant_(aut)_(°c)": "temperatura_orvalho_max_c",
    "temperatura_orvalho_min_na_hora_ant_(aut)_(°c)": "temperatura_orvalho_min_c",
    "umidade_rel_max_na_hora_ant_(aut)_(%)": "umidade_max_porcento",
    "umidade_rel_min_na_hora_ant_(aut)_(%)": "umidade_min_porcento",
    "umidade_relativa_do_ar__horaria_(%)": "umidade_porcento",
    "vento__direção_horaria_(gr)_(°_(gr))": "vento_direcao_graus",
    "vento__rajada_maxima_(m/s)": "vento_rajada_ms",
    "vento__velocidade_horaria_(m/s)": "vento_velocidade_ms"
}


df = df.toDF(*padronizar_colunas)
for antiga_col, nova_col in renames.items():
    df = df.withColumnRenamed(antiga_col, nova_col)
    
df.columns

['regiao',
 'uf',
 'estacao',
 'codigo_(wmo)',
 'latitude',
 'longitude',
 'altitude',
 'data_de_fundacao',
 'data',
 'hora_utc',
 'precipitacao_total_mm',
 'pressao_estacao_mb',
 'pressao_max_mb',
 'pressao_min_mb',
 'radiacao_global_kj_m2',
 'temperatura_seco_c',
 'temperatura_orvalho_c',
 'temperatura_max_c',
 'temperatura_min_c',
 'temperatura_orvalho_max_c',
 'temperatura_orvalho_min_c',
 'umidade_max_porcento',
 'umidade_min_porcento',
 'umidade_porcento',
 'vento_direcao_graus',
 'vento_rajada_ms',
 'vento_velocidade_ms']

# Verificando nulos

In [9]:
for coluna in df.columns:
    print(coluna, df.filter(df[coluna].isNull()).count())

regiao 0
uf 0
estacao 0
codigo_(wmo) 0
latitude 0
longitude 0
altitude 0
data_de_fundacao 0
data 0
hora_utc 0
precipitacao_total_mm 14
pressao_estacao_mb 13
pressao_max_mb 16
pressao_min_mb 17
radiacao_global_kj_m2 4040
temperatura_seco_c 13
temperatura_orvalho_c 13
temperatura_max_c 16
temperatura_min_c 16
temperatura_orvalho_max_c 16
temperatura_orvalho_min_c 17
umidade_max_porcento 16
umidade_min_porcento 16
umidade_porcento 13
vento_direcao_graus 104
vento_rajada_ms 106
vento_velocidade_ms 105


# Arrumando os tipos de schema e adicionando colunas

In [10]:
#from pyspark.sql.types import DecimalType
from pyspark.sql.functions import to_date, col, lpad, month
df = df\
     .withColumn("latitude", F.expr("try_cast(replace(latitude,',','.') as double)"))\
     .withColumn("longitude", F.expr("try_cast(replace(longitude, ',', '.') as double)"))\
     .withColumn("altitude", col("altitude").try_cast("integer"))\
     .withColumn("data", F.to_date("data", "yyyy/MM/dd"))\
     .withColumn("data_br", F.date_format("data", "dd/MM/yyyy"))\
     .withColumn("hora_utc", lpad("hora_utc", 4, "0")).withColumn("hora_utc", F.split("hora_utc", " ")[0])\
     .withColumn("mes", month(col("data")))\
     .withColumn("timestamp_utc", F.concat_ws(" ", F.col("data_br"), F.col("hora_utc")))

# Excluindo Colunas Desnecessárias 

In [11]:
remover_colunas = ["codigo_(wmo)", "data_de_fundacao", "data"]
df = df.drop(*remover_colunas)
df.show(3)

+------+---+----------+------------+-----------+--------+--------+---------------------+------------------+--------------+--------------+---------------------+------------------+---------------------+-----------------+-----------------+-------------------------+-------------------------+--------------------+--------------------+----------------+-------------------+---------------+-------------------+----------+---+---------------+
|regiao| uf|   estacao|    latitude|  longitude|altitude|hora_utc|precipitacao_total_mm|pressao_estacao_mb|pressao_max_mb|pressao_min_mb|radiacao_global_kj_m2|temperatura_seco_c|temperatura_orvalho_c|temperatura_max_c|temperatura_min_c|temperatura_orvalho_max_c|temperatura_orvalho_min_c|umidade_max_porcento|umidade_min_porcento|umidade_porcento|vento_direcao_graus|vento_rajada_ms|vento_velocidade_ms|   data_br|mes|  timestamp_utc|
+------+---+----------+------------+-----------+--------+--------+---------------------+------------------+--------------+--------

# Amplitude Térmica

<p>A <b>amplitude térmica</b> é a diferença entre a máxima e a mínima temperatura de algum local e num determinado período de tempo, mede a variação ao longo do dia</p>
<p>A <b>amplitude térmica anual</b> é calculada pela diferença entre a temperatura média do mês mais quente e a temperatura média do mês mais frio.</p>
<p>A <b>amplitude térmica mensal</b> é calculada pela diferença entre a média do dia mais quente e a temperatura média do dia mais frio.</p>

In [12]:
amplitude_termica = df["mes", "temperatura_max_c", "temperatura_min_c"]

amplitude_termica_mes = amplitude_termica.groupBy(amplitude_termica.mes)\
    .agg(F.min("temperatura_min_c").alias("temp_Min"),\
        F.max("temperatura_max_c").alias("temp_Max"))\
        .withColumn(
            "amplitude_t_mensal",
            F.round(F.col("temp_Max") - F.col("temp_Min"), 1)
        ).sort("mes")

amplitude_termica_mes.show()

+---+--------+--------+------------------+
|mes|temp_Min|temp_Max|amplitude_t_mensal|
+---+--------+--------+------------------+
|  1|    16.7|    30.3|              13.6|
|  2|    17.1|    30.8|              13.7|
|  3|    17.1|    31.5|              14.4|
|  4|    17.2|    30.2|              13.0|
|  5|    14.3|    28.9|              14.6|
|  6|    12.7|    28.2|              15.5|
|  7|    12.5|    28.6|              16.1|
|  8|    12.4|    30.5|              18.1|
|  9|    14.8|    33.3|              18.5|
| 10|    13.4|    33.2|              19.8|
| 11|    16.2|    31.7|              15.5|
| 12|    16.8|    31.2|              14.4|
+---+--------+--------+------------------+



In [13]:
amplitude_termica_anual = amplitude_termica_mes.agg(
    F.round(F.max("temp_Max") - F.min("temp_Min"), 1).alias("amplitude_anual")
)

amplitude_termica_anual.show()

+---------------+
|amplitude_anual|
+---------------+
|           20.9|
+---------------+

